In [22]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# ==========================================
# 1. DATA INGESTION & PREPARATION
# ==========================================
df_linear = pd.read_csv('01-Hg-Linear-scale.csv')
df_angular = pd.read_csv('02-Hg-Angular-scale.csv')
df_unknown = pd.read_csv('03-unknown-Angular-scale.csv')

# Known Grating constants from manual
lines_per_inch = 15000.0
d_cm = 2.54 / lines_per_inch            # Grating spacing in cm
d_nm = d_cm * 1e7                       # Grating spacing in nm (1693.33 nm)
f_cm = 21.25                            # Focal length in cm
R_cm = 2.0 * f_cm                       # Radius of curvature R = 42.50 cm
W_cm = 1.8                              # Width of ruled grating area in cm
N_total = W_cm * (lines_per_inch / 2.54)# Total rulings (~10,630 lines)

# Mercury Linear Scale Data
x_o1 = df_linear['Reading (cm)'].iloc[0:8].values
lam_o1 = df_linear['Wavelength (nm)'].iloc[0:8].values

x_o2 = df_linear['Reading (cm)'].iloc[8:16].values
lam_o2 = df_linear['Wavelength (nm)'].iloc[8:16].values

# Mercury Angular Scale Data
# Reference line is Green (546.1 nm, index 3).
# Lines with shorter wavelengths (indices 0, 1, 2) lie at negative angular deviations.
signs = np.array([-1.0, -1.0, -1.0, 0.0, 1.0, 1.0, 1.0, 1.0])
theta_o1 = signs * df_angular['Angular Scale (degrees)'].iloc[0:8].values
theta_o2 = signs * df_angular['Angular Scale (degrees)'].iloc[8:16].values

# Unknown Source Linear Data (Sodium Vapour Lamp)
x_unk_o1 = df_unknown['Full Reading (cm)'].iloc[0:2].values
x_unk_o2 = np.sort(df_unknown['Full Reading (cm)'].iloc[2:4].values)

# ==========================================
# 2. REGRESSION & DISPERSION ANALYSIS
# ==========================================
# Linear scale fit: lambda = m*x + c
fit_lin_o1 = stats.linregress(x_o1, lam_o1)
fit_lin_o2 = stats.linregress(x_o2, lam_o2)

# High-precision linear fit (first 7 prominent lines up to 623.4 nm)
fit_lin7_o1 = stats.linregress(x_o1[:7], lam_o1[:7])
fit_lin7_o2 = stats.linregress(x_o2[:7], lam_o2[:7])

# Angular scale fit: lambda = m*theta + c
fit_ang_o1 = stats.linregress(theta_o1, lam_o1)
fit_ang_o2 = stats.linregress(theta_o2, lam_o2)

# High-precision angular fit (first 7 prominent lines)
fit_ang7_o1 = stats.linregress(theta_o1[:7], lam_o1[:7])
fit_ang7_o2 = stats.linregress(theta_o2[:7], lam_o2[:7])

# Ratio tests: (Order 1 / Order 2)
ratio_lin = fit_lin_o1.slope / fit_lin_o2.slope
ratio_lin_err = ratio_lin * np.sqrt((fit_lin_o1.stderr/fit_lin_o1.slope)**2 + (fit_lin_o2.stderr/fit_lin_o2.slope)**2)

ratio_ang = fit_ang_o1.slope / fit_ang_o2.slope
ratio_ang_err = ratio_ang * np.sqrt((fit_ang_o1.stderr/fit_ang_o1.slope)**2 + (fit_ang_o2.stderr/fit_ang_o2.slope)**2)

# ==========================================
# 3. UNKNOWN WAVELENGTH DETERMINATION
# ==========================================
# Using high-precision calibration curve:
lam_unk_o1 = fit_lin7_o1.slope * x_unk_o1 + fit_lin7_o1.intercept
lam_unk_o2 = fit_lin7_o2.slope * x_unk_o2 + fit_lin7_o2.intercept

# Standard errors on predicted values
err_pred_o1 = np.sqrt(fit_lin7_o1.intercept_stderr**2 + (x_unk_o1 * fit_lin7_o1.stderr)**2 + (fit_lin7_o1.slope * 0.005)**2)
err_pred_o2 = np.sqrt(fit_lin7_o2.intercept_stderr**2 + (x_unk_o2 * fit_lin7_o2.stderr)**2 + (fit_lin7_o2.slope * 0.005)**2)

# ==========================================
# 4. PLOTTING PUBLICATION-GRADE FIGURES
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axs = plt.subplots(2, 2, figsize=(14, 11), dpi=300)
c1, c2 = '#1f77b4', '#d62728'

# (a) Linear Scale Calibration
ax = axs[0, 0]
ax.errorbar(x_o1, lam_o1, xerr=0.005, fmt='o', color=c1, label=f'Hg Order 1 ($n=1$)\nSlope = {fit_lin_o1.slope:.2f} $\\pm$ {fit_lin_o1.stderr:.2f} nm/cm', capsize=3)
xg1 = np.linspace(min(x_o1)-0.5, max(x_o1)+0.5, 100)
ax.plot(xg1, fit_lin_o1.slope * xg1 + fit_lin_o1.intercept, color=c1, linestyle='--')

ax.errorbar(x_o2, lam_o2, xerr=0.005, fmt='s', color=c2, label=f'Hg Order 2 ($n=2$)\nSlope = {fit_lin_o2.slope:.2f} $\\pm$ {fit_lin_o2.stderr:.2f} nm/cm', capsize=3)
xg2 = np.linspace(min(x_o2)-0.5, max(x_o2)+0.5, 100)
ax.plot(xg2, fit_lin_o2.slope * xg2 + fit_lin_o2.intercept, color=c2, linestyle='--')

ax.scatter(x_unk_o1, fit_lin_o1.slope * x_unk_o1 + fit_lin_o1.intercept, color='green', marker='^', s=85, zorder=5, label=f'Unknown (O1): $\\lambda \\approx$ {np.mean(fit_lin_o1.slope * x_unk_o1 + fit_lin_o1.intercept):.1f} nm')
ax.scatter(x_unk_o2, fit_lin_o2.slope * x_unk_o2 + fit_lin_o2.intercept, color='purple', marker='v', s=85, zorder=5, label=f'Unknown (O2): $\\lambda \\approx$ {np.mean(fit_lin_o2.slope * x_unk_o2 + fit_lin_o2.intercept):.1f} nm')
ax.set_title('(a) Calibration Curve: Wavelength vs Linear Position ($x$)', fontsize=12, fontweight='bold')
ax.set_xlabel('Linear Position $x$ (cm)', fontsize=11)
ax.set_ylabel('Wavelength $\\lambda$ (nm)', fontsize=11)
ax.legend(frameon=True, fontsize=8.5, loc='upper left')

# (b) Angular Scale Calibration
ax = axs[0, 1]
ax.errorbar(theta_o1, lam_o1, xerr=0.1, fmt='o', color=c1, label=f'Hg Order 1 ($n=1$)\nSlope = {fit_ang_o1.slope:.2f} $\\pm$ {fit_ang_o1.stderr:.2f} nm/$^\\circ$', capsize=3)
tg1 = np.linspace(min(theta_o1)-0.5, max(theta_o1)+0.5, 100)
ax.plot(tg1, fit_ang_o1.slope * tg1 + fit_ang_o1.intercept, color=c1, linestyle='--')

ax.errorbar(theta_o2, lam_o2, xerr=0.1, fmt='s', color=c2, label=f'Hg Order 2 ($n=2$)\nSlope = {fit_ang_o2.slope:.2f} $\\pm$ {fit_ang_o2.stderr:.2f} nm/$^\\circ$', capsize=3)
tg2 = np.linspace(min(theta_o2)-0.5, max(theta_o2)+0.5, 100)
ax.plot(tg2, fit_ang_o2.slope * tg2 + fit_ang_o2.intercept, color=c2, linestyle='--')

ax.axvline(0, color='gray', linestyle=':', alpha=0.7, label='Reference line (Green 546.1 nm)')
ax.set_title('(b) Calibration Curve: Wavelength vs Angular Separation ($\\theta$)', fontsize=12, fontweight='bold')
ax.set_xlabel('Angular Separation $\\theta$ ($^\\circ$)', fontsize=11)
ax.set_ylabel('Wavelength $\\lambda$ (nm)', fontsize=11)
ax.legend(frameon=True, fontsize=8.5, loc='upper left')

# (c) High-Precision Prominent Line Calibration & Sodium Determination
ax = axs[1, 0]
ax.plot(x_o1[:7], lam_o1[:7], 'o', color=c1, label=f'O1 (Prominent lines): {fit_lin7_o1.slope:.2f} nm/cm ($R^2={fit_lin7_o1.rvalue**2:.4f}$)')
ax.plot(xg1, fit_lin7_o1.slope * xg1 + fit_lin7_o1.intercept, color=c1)

ax.plot(x_o2[:7], lam_o2[:7], 's', color=c2, label=f'O2 (Prominent lines): {fit_lin7_o2.slope:.2f} nm/cm ($R^2={fit_lin7_o2.rvalue**2:.4f}$)')
ax.plot(xg2, fit_lin7_o2.slope * xg2 + fit_lin7_o2.intercept, color=c2)

ax.scatter(x_unk_o1, lam_unk_o1, color='green', marker='^', s=90, zorder=5, label=f'Na Doublet O1: {lam_unk_o1[0]:.1f}, {lam_unk_o1[1]:.1f} nm')
ax.scatter(x_unk_o2, lam_unk_o2, color='purple', marker='v', s=90, zorder=5, label=f'Na Doublet O2: {lam_unk_o2[0]:.1f}, {lam_unk_o2[1]:.1f} nm')
ax.axhline(589.3, color='goldenrod', linestyle=':', linewidth=2, label='Accepted Na doublet mean (589.3 nm)')
ax.set_title('(c) High-Precision Fit & Na Doublet Determination', fontsize=12, fontweight='bold')
ax.set_xlabel('Linear Position $x$ (cm)', fontsize=11)
ax.set_ylabel('Wavelength $\\lambda$ (nm)', fontsize=11)
ax.legend(frameon=True, fontsize=8.5, loc='upper left')

# (d) Verification of Inverse Order Law
ax = axs[1, 1]
inv_n = np.array([1.0, 0.5])
slopes_lin = [fit_lin_o1.slope, fit_lin_o2.slope]
slopes_ang = [fit_ang_o1.slope, fit_ang_o2.slope]
ax.errorbar(inv_n, slopes_lin, yerr=[fit_lin_o1.stderr, fit_lin_o2.stderr], fmt='o-', color='#2ca02c', capsize=5, linewidth=2, label='$(d\\lambda/dx)$ vs $1/n$ [nm/cm]')
ax.errorbar(inv_n, slopes_ang, yerr=[fit_ang_o1.stderr, fit_ang_o2.stderr], fmt='s-', color='#9467bd', capsize=5, linewidth=2, label='$(d\\lambda/d\\theta)$ vs $1/n$ [nm/$^\\circ$]')
ax.plot([0, 1.1], [0, slopes_lin[0]*1.1], '--', color='#2ca02c', alpha=0.5)
ax.plot([0, 1.1], [0, slopes_ang[0]*1.1], '--', color='#9467bd', alpha=0.5)
ax.set_title('(d) Verification of Inverse Order Law ($(d\\lambda/dp) \\propto 1/n$)', fontsize=12, fontweight='bold')
ax.set_xlabel('Inverse Spectral Order $1/n$', fontsize=11)
ax.set_ylabel('Dispersion Slope ($d\\lambda/dx$ or $d\\lambda/d\\theta$)', fontsize=11)
ax.set_xlim(0, 1.15)
ax.set_ylim(0, 52)
ax.legend(frameon=True, fontsize=9.5, loc='upper left')

plt.tight_layout()
plt.savefig('concave_grating_analysis.png', dpi=300)
plt.show()